# 01 — Setup Google Drive, Ekstraksi EXIF Foto HP, & Download Dataset
**Runtime yang disarankan:** CPU saja (agar tidak membuang kuota GPU Colab Free).

Notebook ini mengerjakan:
1. Mount Google Drive untuk menyimpan dataset, checkpoint, dan log secara permanen.
2. Ekstraksi **EXIF / metadata** dari folder 40+ foto HP Anda ke tabel CSV (`exif_metadata.csv`) sebagai bukti keaslian test set.
3. Download dataset utama (data latih + test set asli) dan **test set buatan Anda** dari Roboflow dalam 2 format sekaligus:
   - Format **YOLOv8** (untuk Model 1: YOLOv8-seg)
   - Format **COCO Segmentation** (untuk Model 2: Mask R-CNN & Model 3: Mask2Former)
CUSTOM_WORKSPACE = "cdaaptnia"
CUSTOM_PROJECT   = "test-hp-pakaian"
CUSTOM_VERSION   = 1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/cpmk3-instance-segmentation'
for sub in ['datasets', 'raw_hp_photos', 'runs', 'logs', 'exports']:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)
print('Folder proyek siap:', PROJECT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder proyek siap: /content/drive/MyDrive/cpmk3-instance-segmentation


## Bagian A: Ekstraksi EXIF / Metadata Foto HP (Test Set Sendiri)
Upload minimal **40 foto asli dari kamera HP Anda** (jangan lewat WhatsApp karena WA menghapus EXIF; upload langsung dari HP ke Google Drive) ke folder:
`/content/drive/MyDrive/CPMK3_Instance_Segmentation/raw_hp_photos/`

*(Opsional: pisahkan dalam subfolder `normal/` dan `sulit/` agar otomatis tercatat kategorinya).*


In [ ]:
!pip install -q Pillow pandas

import glob
import pandas as pd
from PIL import Image, ExifTags

PHOTO_DIR = '/content/drive/MyDrive/Indoor_datasets'
img_exts = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
photo_paths = []
for ext in img_exts:
    photo_paths.extend(glob.glob(os.path.join(PHOTO_DIR, '**', ext), recursive=True))

print(f'Total foto ditemukan: {len(photo_paths)}')
if len(photo_paths) < 40:
    print('PERINGATAN: Jumlah foto masih kurang dari 40 (syarat minimal 40 foto, min. 10 kondisi sulit).')

TAG_MAP = {v: k for k, v in ExifTags.TAGS.items()}
TARGET_TAGS = [
    'Make', 'Model', 'DateTimeOriginal', 'ExposureTime',
    'FNumber', 'ISOSpeedRatings', 'FocalLength', 'Orientation'
]

rows = []
for p in sorted(photo_paths):
    rel_path = os.path.relpath(p, PHOTO_DIR)
    folder = os.path.dirname(rel_path) or 'root'
    try:
        with Image.open(p) as img:
            w, h = img.size
            exif_raw = img._getexif() or {}
            row = {
                'filename': os.path.basename(p),
                'subfolder': folder,
                'width': w,
                'height': h,
                'format': img.format,
            }
            for tag_name in TARGET_TAGS:
                tag_id = TAG_MAP.get(tag_name)
                val = exif_raw.get(tag_id, None)
                row[tag_name] = str(val) if val is not None else ''
            rows.append(row)
    except Exception as e:
        rows.append({'filename': os.path.basename(p), 'subfolder': folder, 'error': str(e)})

df_exif = pd.DataFrame(rows)
exif_csv = os.path.join(PROJECT_DIR, 'logs', 'exif_metadata.csv')
df_exif.to_csv(exif_csv, index=False)
print('EXIF metadata disimpan ke:', exif_csv)
df_exif.head(10)


Total foto ditemukan: 41
EXIF metadata disimpan ke: /content/drive/MyDrive/cpmk3-instance-segmentation/logs/exif_metadata.csv


,filename,subfolder,width,height,format,Make,Model,DateTimeOriginal,ExposureTime,FNumber,ISOSpeedRatings,FocalLength,Orientation
0,WhatsApp Image 2026-11-19 at 20.06.51 (1).jpeg,root,1472,1472,JPEG,,,,,,,,
1,WhatsApp Image 2026-11-24 at 15.30.51.jpeg,root,1472,1472,JPEG,,,,,,,,
2,WhatsApp Image 2026-11-24 at 15.30.52 (1).jpeg,root,1472,1472,JPEG,,,,,,,,
3,WhatsApp Image 2026-11-24 at 15.30.52 (2).jpeg,root,1472,1472,JPEG,,,,,,,,
4,WhatsApp Image 2026-11-24 at 15.30.53 (1).jpeg,root,1472,1472,JPEG,,,,,,,,
5,WhatsApp Image 2026-11-24 at 15.30.53 (2).jpeg,root,1472,1472,JPEG,,,,,,,,
6,WhatsApp Image 2026-11-24 at 15.30.53.jpeg,root,1472,1472,JPEG,,,,,,,,
7,WhatsApp Image 2026-11-24 at 15.30.56 (1).jpeg,root,1472,1472,JPEG,,,,,,,,
8,WhatsApp Image 2026-11-24 at 15.30.56 (3).jpeg,root,1472,1472,JPEG,,,,,,,,
9,WhatsApp Image 2026-11-24 at 15.30.56.jpeg,root,1472,1472,JPEG,,,,,,,,


## Bagian B: Download Dataset dari Roboflow
Isi API Key, Workspace, Project, dan Version Roboflow Anda di bawah ini.
- **Dataset Utama**: dataset latih + validasi + test set asli (misal dataset baju/jaket/kaos atau Panoptic Indoor Segmentation v5).
- **Dataset Test HP**: proyek Roboflow berisi 40+ foto HP Anda yang sudah Anda anotasi dengan **nama kelas yang sama persis**.


In [ ]:
import yaml
import os
!pip install -q roboflow

from roboflow import Roboflow

# === ISI API KEY ROBOFLOW ANDA ===
import getpass
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY") or getpass.getpass("Masukkan Roboflow API Key: ")

# 1. Dataset Utama: Panoptic Indoor Segmentation v5
MAIN_WORKSPACE = "panopticindoor"
MAIN_PROJECT   = "panoptic-indoor-segmentation"
MAIN_VERSION   = 5

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Download Dataset Utama (format YOLOv8 & COCO)
main_ver = rf.workspace(MAIN_WORKSPACE).project(MAIN_PROJECT).version(MAIN_VERSION)
main_yolo_dir = os.path.join(PROJECT_DIR, 'datasets', 'main_yolo')
main_coco_dir = os.path.join(PROJECT_DIR, 'datasets', 'main_coco')
main_ver.download("yolov8", location=main_yolo_dir, overwrite=True)
main_ver.download("coco-segmentation", location=main_coco_dir, overwrite=True)

# 2. Dataset Test Set HP Ruangan (aktifkan nanti setelah 40 foto ruangan selesai dianotasi)
# CUSTOM_WORKSPACE = "cdaaptnia"
# CUSTOM_PROJECT   = "test-hp-ruangan"
# CUSTOM_VERSION   = 1
# custom_ver = rf.workspace(CUSTOM_WORKSPACE).project(CUSTOM_PROJECT).version(CUSTOM_VERSION)
# custom_ver.download("yolov8", location=os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_yolo'), overwrite=True)
# custom_ver.download("coco-segmentation", location=os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_coco'), overwrite=True)

# Cek daftar kelas Panoptic Indoor v5
with open(os.path.join(main_yolo_dir, 'data.yaml')) as f:
    cfg = yaml.safe_load(f)
print("\nPanoptic Indoor v5 berhasil diunduh!")
print("Jumlah kelas:", len(cfg['names']))
print("Daftar kelas:", cfg['names'])

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/drive/MyDrive/cpmk3-instance-segmentation/datasets/main_yolo in yolov8:: 100%|██████████| 1856/1856 [00:19<00:00, 97.08it/s] 


Extracting Dataset Version Zip to /content/drive/MyDrive/cpmk3-instance-segmentation/datasets/main_coco in coco-segmentation:: 100%|██████████| 930/930 [00:09<00:00, 97.69it/s]



Panoptic Indoor v5 berhasil diunduh!
Jumlah kelas: 74
Daftar kelas: ['backpack', 'bed', 'blanket', 'book', 'bottle', 'bowl', 'box', 'cabinet-merged', 'cardboard', 'ceiling-merged', 'cell phone', 'chair', 'clock', 'couch', 'counter', 'cup', 'curtain', 'dining table', 'door-stuff', 'fire-extinguisher', 'floor-other-merged', 'floor-wood', 'flower', 'food-other-merged', 'fork', 'hair drier', 'handbag', 'keyboard', 'knife', 'laptop', 'light', 'microwave', 'mirror-stuff', 'mouse', 'oven', 'pan', 'paper', 'paper-merged', 'person', 'pillow', 'plate', 'plug', 'potted plant', 'refrigerator', 'remote', 'rug-merged', 'scissor', 'scissors', 'shelf', 'sink', 'spoon', 'st-', 'stairs', 'stove-merged', 'table-merged', 'teddy bear', 'toaster', 'toilet', 'toothbrush', 'towel', 'toy', 'trash-bin', 'tree-merged', 'tv', 'vase', 'wall', 'wall-brick', 'wall-other-merged', 'wall-stone', 'wall-tile', 'wall-wood', 'wallet', 'window-blind', 'window-other']


## Bagian C: Verifikasi Kesesuaian Kelas (Main vs Test HP)
Pastikan urutan & nama kelas antara dataset utama dan test set HP **100% sama** agar evaluasi tidak keliru.
!pip install roboflow


In [ ]:
import os, glob, yaml
from roboflow import Roboflow

PROJECT_DIR = '/content/drive/MyDrive/cpmk3-instance-segmentation'

# === ISI 2 BARIS INI ===
import getpass
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY") or getpass.getpass("Masukkan Roboflow API Key: ")
CUSTOM_PROJECT   = "test-hp-ruangan"
CUSTOM_WORKSPACE = "cdaaptnia"
CUSTOM_VERSION   = 2

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
custom_ver = rf.workspace(CUSTOM_WORKSPACE).project(CUSTOM_PROJECT).version(CUSTOM_VERSION)

custom_yolo_dir = os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_yolo')
custom_coco_dir = os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_coco')
custom_ver.download("yolov8", location=custom_yolo_dir, overwrite=True)
custom_ver.download("coco-segmentation", location=custom_coco_dir, overwrite=True)

# === REMAP OTOMATIS ID KELAS CUSTOM_HP_YOLO -> 74 KELAS RESMI PANOPTIC INDOOR V5 ===
with open(os.path.join(PROJECT_DIR, 'datasets', 'main_yolo', 'data.yaml')) as f:
    main_names = yaml.safe_load(f)['names']
with open(os.path.join(custom_yolo_dir, 'data.yaml')) as f:
    custom_cfg = yaml.safe_load(f)
custom_names = custom_cfg['names']

main_name_to_id = {name: idx for idx, name in enumerate(main_names)}
remap = {}
unmatched = []
for old_id, name in enumerate(custom_names):
    if name in main_name_to_id:
        remap[old_id] = main_name_to_id[name]
    else:
        unmatched.append(name)

print(f"Kelas cocok: {len(remap)}/{len(custom_names)} | Tidak cocok: {unmatched}")

# Tulis ulang semua file label .txt memakai ID resmi v5
label_files = glob.glob(os.path.join(custom_yolo_dir, '**', 'labels', '*.txt'), recursive=True)
for lf in label_files:
    new_lines = []
    with open(lf, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            old_id = int(parts[0])
            if old_id in remap:
                parts[0] = str(remap[old_id])
                new_lines.append(' '.join(parts))
    with open(lf, 'w') as f:
        f.write('\n'.join(new_lines) + ('\n' if new_lines else ''))

# Update data.yaml custom_hp_yolo agar pakai daftar 74 kelas resmi v5
custom_cfg['names'] = main_names
custom_cfg['nc'] = len(main_names)
with open(os.path.join(custom_yolo_dir, 'data.yaml'), 'w') as f:
    yaml.safe_dump(custom_cfg, f, sort_keys=False)

print(f"SELESAI! {len(label_files)} file label berhasil disamakan ke 74 kelas Panoptic Indoor v5.")

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 0.1%
Version export complete for yolov8 format



Extracting Dataset Version Zip to /content/drive/MyDrive/cpmk3-instance-segmentation/datasets/custom_hp_yolo in yolov8:: 100%|██████████| 83/83 [00:01<00:00, 82.19it/s]


Exporting format coco-segmentation in progress : 95.0%
Version export complete for coco-segmentation format



Extracting Dataset Version Zip to /content/drive/MyDrive/cpmk3-instance-segmentation/datasets/custom_hp_coco in coco-segmentation:: 100%|██████████| 43/43 [00:00<00:00, 75.77it/s]


Kelas cocok: 27/27 | Tidak cocok: []
SELESAI! 40 file label berhasil disamakan ke 74 kelas Panoptic Indoor v5.


In [ ]:
import yaml, json

with open(os.path.join(PROJECT_DIR, 'datasets', 'main_yolo', 'data.yaml')) as f:
    main_yaml = yaml.safe_load(f)
with open(os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_yolo', 'data.yaml')) as f:
    custom_yaml = yaml.safe_load(f)

print("Kelas Dataset Utama :", main_yaml.get('names'))
print("Kelas Test Set HP   :", custom_yaml.get('names'))
assert main_yaml.get('names') == custom_yaml.get('names'), (
    "PERINGATAN: Nama/urutan kelas tidak sama! Perbaiki kelas di Roboflow sebelum training."
)
print("OK! Daftar kelas cocok 100%.")


Kelas Dataset Utama : ['backpack', 'bed', 'blanket', 'book', 'bottle', 'bowl', 'box', 'cabinet-merged', 'cardboard', 'ceiling-merged', 'cell phone', 'chair', 'clock', 'couch', 'counter', 'cup', 'curtain', 'dining table', 'door-stuff', 'fire-extinguisher', 'floor-other-merged', 'floor-wood', 'flower', 'food-other-merged', 'fork', 'hair drier', 'handbag', 'keyboard', 'knife', 'laptop', 'light', 'microwave', 'mirror-stuff', 'mouse', 'oven', 'pan', 'paper', 'paper-merged', 'person', 'pillow', 'plate', 'plug', 'potted plant', 'refrigerator', 'remote', 'rug-merged', 'scissor', 'scissors', 'shelf', 'sink', 'spoon', 'st-', 'stairs', 'stove-merged', 'table-merged', 'teddy bear', 'toaster', 'toilet', 'toothbrush', 'towel', 'toy', 'trash-bin', 'tree-merged', 'tv', 'vase', 'wall', 'wall-brick', 'wall-other-merged', 'wall-stone', 'wall-tile', 'wall-wood', 'wallet', 'window-blind', 'window-other']
Kelas Test Set HP   : ['backpack', 'bed', 'blanket', 'book', 'bottle', 'bowl', 'box', 'cabinet-merged',